# 02 · Summary metrics and confidence intervals

Compute per-seed mean loss and step-300 loss from all 1,500 telemetry rows; compare to `summary.json`. CI is a two-sided t interval across five seeds. Trajectory bands are pointwise across seed-level records, not independent-run confidence claims.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
by_seed=df.sort_values("step").groupby("seed").agg(avg_loss=("loss","mean"),final_loss=("loss","last"))
print(by_seed)
def ci_stats(x):
    x=np.asarray(x); sd=x.std(ddof=1); ci=2.7764451051977987*sd/np.sqrt(len(x)) # t(.975,4)
    return pd.Series({"mean":x.mean(),"sample_sd":sd,"95%_halfwidth":ci})
emp=pd.DataFrame({k:ci_stats(by_seed[k]) for k in ["avg_loss","final_loss"]}).T
print("Empirical seed-level statistics:\n",emp)
print("summary.json:\n",json.dumps(summary,indent=2))
print("Differences (empirical mean/std/CI halfwidth - JSON fields):")
for key,col in [("avg_loss","avg_loss"),("final_loss","final_loss")]:
    print(key,emp.loc[col,["mean","sample_sd","95%_halfwidth"]].to_numpy()-np.array([summary[key]["mean"],summary[key]["std"],summary[key]["ci95"]]))

In [ ]:
traj=df.groupby("step").loss.agg(mean="mean",sd="std",n="count").reset_index()
traj["ci"]=2.7764451051977987*traj.sd/np.sqrt(traj.n)
fig,ax=plt.subplots(figsize=(9,4)); ax.plot(traj.step,traj["mean"],color="#263f73"); ax.fill_between(traj.step,traj["mean"]-traj.ci,traj["mean"]+traj.ci,alpha=.2)
ax.set(xlabel="step",ylabel="loss",title="Across-seed loss trajectory (pointwise 95% t CI)"); plt.show()
assert traj.n.eq(5).all(); print("Trajectory rows:",len(traj),"first/last mean:",traj.iloc[0]["mean"],traj.iloc[-1]["mean"])

**Interpretation:** JSON’s average-loss mean matches the pooled CSV mean, which is not the same as a mean of seed-level averages' uncertainty. The explicit differences above reveal whether the JSON spread is across seeds or otherwise calculated.